# Cluster Training Example

Run this notebook from a GPU node, following the instructions in `01-amarel-setup.md`.

## Setup

Add basic imports.

In [1]:
import time

import torch
from torch import nn

## Check to Ensure GPUs are Available

Here we check that GPUs are available and print the GPUs assigned to us.

In [ ]:
# Require an NVIDIA GPU
assert torch.cuda.is_available(), "CUDA is unavailable"
device = torch.device("cuda")

print("PyTorch:", torch.__version__, flush=True)
print("GPU:", torch.cuda.get_device_name(0), flush=True)


## Generate Example Data

Generate a simple synthetic dataset for example purposes.

In [ ]:
torch.manual_seed(42)

x = torch.randn(10000, 100, device=device)
y = (x[:, :10].sum(dim=1) > 0).long()

## Define a Simple Network

Here we define a simple model with an optimizer and a loss function.

In [ ]:
model = nn.Sequential(
    nn.Linear(100, 256),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, 2),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

## Train the Model

Train the model on the cluster!

In [ ]:
torch.cuda.synchronize()
start = time.perf_counter()

for epoch in range(100):
    optimizer.zero_grad()

    output = model(x)
    loss = loss_fn(output, y)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}", flush=True)

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

print(f"Training time: {elapsed:.2f} seconds")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.3f} GB")

# Save the trained model
torch.save(model.state_dict(), "model.pt")
print("Training complete!")